# Collaborative filtering walkthrough

Portfolio refactor of the original Steam coursework. This notebook uses invented data and the same modules as the dashboard; no Steam data is downloaded. Scores are illustrative, not measured preferences. Run from the repository or notebooks directory.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / "steam_rec").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from steam_rec.data import demo_matrix
from steam_rec.model import predict, recommend
from steam_rec.evaluate import split_observed, evaluate
frame = demo_matrix()
print(frame.shape)
frame.head()

## Hold out observations, not entire users

A full-size aligned mask hides one or more interactions but keeps history for each evaluated player. Similarities and means are derived from training values only.

In [ ]:
train, held = split_observed(frame, seed=42)
assert train[held].isna().all().all()
print("Held-out observations:", int(held.to_numpy().sum()))
metrics, settings = evaluate(frame)
print(settings)
metrics

## Compare similarity choices

Use the same split across all methods. This is a validation demonstration, not a separate final-test result. Lower regression error need not imply better ranking.

In [ ]:
for metric in ["cosine", "jaccard", "pearson", "mixed"]:
    table, _ = evaluate(frame, metric=metric)
    print(metric)
    print(table.to_string(index=False))

## Recommend using the full available history

Evaluation is finished. For interactive recommendations, fit the full matrix and exclude already observed games. This fit is not used to calculate holdout metrics.

In [ ]:
scores = predict(frame)["Hybrid"]
recommend(frame, scores, frame.index[0], top_n=10)